In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import os

# Create Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("hospital-bronze") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "8")  # optimised for local

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Define paths
PROJECT_BASE = os.path.expanduser("~/data-engineering-pathway/projects/hospital-lakehouse")
DATA_PATH    = f"{PROJECT_BASE}/data/healthcare_raw.csv"
DELTA_BASE   = f"{PROJECT_BASE}/delta"

print(f"Spark version : {spark.version}")
print(f"Data path     : {DATA_PATH}")
print(f"Delta base    : {DELTA_BASE}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/25 14:30:29 WARN Utils: Your hostname, M-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.4 instead (on interface en0)
26/04/25 14:30:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/malone/data-engineering-pathway/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/malone/.ivy2.5.2/cache
The jars for the packages stored in: /Users/malone/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9d257c3d-696d-4585-8102-f5c3adc59631;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0

Spark version : 4.1.1
Data path     : /Users/malone/data-engineering-pathway/projects/hospital-lakehouse/data/healthcare_raw.csv
Delta base    : /Users/malone/data-engineering-pathway/projects/hospital-lakehouse/delta


In [2]:
# Load raw CSV — no transformations, Bronze principle
raw_df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

print(f"Raw rows     : {raw_df.count()}")
print(f"Raw columns  : {len(raw_df.columns)}")
print(f"\nSchema:")
raw_df.printSchema()
print(f"\nSample row:")
raw_df.show(1, truncate=False, vertical=True)

Raw rows     : 55500
Raw columns  : 15

Schema:
root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Blood Type: string (nullable = true)
 |-- Medical Condition: string (nullable = true)
 |-- Date of Admission: date (nullable = true)
 |-- Doctor: string (nullable = true)
 |-- Hospital: string (nullable = true)
 |-- Insurance Provider: string (nullable = true)
 |-- Billing Amount: double (nullable = true)
 |-- Room Number: integer (nullable = true)
 |-- Admission Type: string (nullable = true)
 |-- Discharge Date: date (nullable = true)
 |-- Medication: string (nullable = true)
 |-- Test Results: string (nullable = true)


Sample row:
-RECORD 0--------------------------------
 Name               | Bobby JacksOn      
 Age                | 30                 
 Gender             | Male               
 Blood Type         | B-                 
 Medical Condition  | Cancer             
 Date of Admission  | 2024-01-31       

In [3]:
from pyspark.sql.functions import col, count, when, countDistinct

print("=" * 60)
print("BRONZE PROFILE REPORT")
print("=" * 60)

# Null counts
print("\nNull counts per column:")
null_counts = raw_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in raw_df.columns
]).collect()[0].asDict()

for col_name, null_count in null_counts.items():
    status = f"{null_count} nulls" if null_count > 0 else "clean"
    print(f"  {col_name:<25} {status}")

# Distinct value counts for categorical columns
print("\nDistinct values in key columns:")
for c in ["Gender", "Admission Type", "Medical Condition", 
          "Insurance Provider", "Test Results", "Medication"]:
    col_clean = c.replace(" ", "_")
    n = raw_df.select(countDistinct(f"`{c}`")).collect()[0][0]
    print(f"  {c:<25} {n} distinct values")

# Admission type breakdown
print("\nAdmission type breakdown:")
raw_df.groupBy("`Admission Type`").count().orderBy("count", ascending=False).show()

# Medical condition breakdown
print("Medical condition breakdown:")
raw_df.groupBy("`Medical Condition`").count().orderBy("count", ascending=False).show()

print("=" * 60)

BRONZE PROFILE REPORT

Null counts per column:
  Name                      clean
  Age                       clean
  Gender                    clean
  Blood Type                clean
  Medical Condition         clean
  Date of Admission         clean
  Doctor                    clean
  Hospital                  clean
  Insurance Provider        clean
  Billing Amount            clean
  Room Number               clean
  Admission Type            clean
  Discharge Date            clean
  Medication                clean
  Test Results              clean

Distinct values in key columns:
  Gender                    2 distinct values
  Admission Type            3 distinct values
  Medical Condition         6 distinct values
  Insurance Provider        5 distinct values
  Test Results              3 distinct values
  Medication                5 distinct values

Admission type breakdown:
+--------------+-----+
|Admission Type|count|
+--------------+-----+
|      Elective|18655|
|        Urgent

In [7]:
from pyspark.sql.functions import monotonically_increasing_id

# Split into patients — demographics + PII
# admission_id links patients to admissions
bronze_patients = raw_df \
    .withColumn("admission_id", monotonically_increasing_id()) \
    .select(
        col("admission_id"),
        col("Name").alias("patient_name"),       # PII — masked in Silver
        col("Age").alias("age"),
        col("Gender").alias("gender"),
        col("`Blood Type`").alias("blood_type")
    )

# Write as Delta table
bronze_patients.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{DELTA_BASE}/bronze/patients")

print(f"bronze_patients written")
print(f"  Rows    : {bronze_patients.count()}")
print(f"  Columns : {bronze_patients.columns}")
bronze_patients.show(5)

bronze_patients written
  Rows    : 55500
  Columns : ['admission_id', 'patient_name', 'age', 'gender', 'blood_type']
+------------+-------------+---+------+----------+
|admission_id| patient_name|age|gender|blood_type|
+------------+-------------+---+------+----------+
|           0|Bobby JacksOn| 30|  Male|        B-|
|           1| LesLie TErRy| 62|  Male|        A+|
|           2|  DaNnY sMitH| 76|Female|        A-|
|           3| andrEw waTtS| 28|Female|        O+|
|           4|adrIENNE bEll| 43|Female|       AB+|
+------------+-------------+---+------+----------+
only showing top 5 rows


In [8]:
# Split into admissions — clinical + operational + financial
from pyspark.sql.functions import monotonically_increasing_id, col
bronze_admissions = raw_df \
    .withColumn("admission_id", monotonically_increasing_id()) \
    .select(
        col("admission_id"),
        col("`Medical Condition`").alias("medical_condition"),
        col("`Date of Admission`").alias("date_of_admission"),
        col("Doctor").alias("doctor"),
        col("Hospital").alias("hospital"),
        col("`Insurance Provider`").alias("insurance_provider"),
        col("`Billing Amount`").alias("billing_amount"),
        col("`Room Number`").alias("room_number"),
        col("`Admission Type`").alias("admission_type"),
        col("`Discharge Date`").alias("discharge_date"),
        col("Medication").alias("medication"),
        col("`Test Results`").alias("test_results")
    )

# Write as Delta table
bronze_admissions.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{DELTA_BASE}/bronze/admissions")

print(f"bronze_admissions written")
print(f"  Rows    : {bronze_admissions.count()}")
print(f"  Columns : {bronze_admissions.columns}")
bronze_admissions.show(5, truncate=False)

bronze_admissions written
  Rows    : 55500
  Columns : ['admission_id', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results']
+------------+-----------------+-----------------+----------------+--------------------------+------------------+------------------+-----------+--------------+--------------+-----------+------------+
|admission_id|medical_condition|date_of_admission|doctor          |hospital                  |insurance_provider|billing_amount    |room_number|admission_type|discharge_date|medication |test_results|
+------------+-----------------+-----------------+----------------+--------------------------+------------------+------------------+-----------+--------------+--------------+-----------+------------+
|0           |Cancer           |2024-01-31       |Matthew Smith   |Sons and Miller           |Blue Cross        |18856.281305978155|328       

In [9]:
# Reload from Delta to confirm persistence
patients_reloaded   = spark.read.format("delta").load(f"{DELTA_BASE}/bronze/patients")
admissions_reloaded = spark.read.format("delta").load(f"{DELTA_BASE}/bronze/admissions")

print("=" * 60)
print("BRONZE LAYER — HOSPITAL COMPLETE")
print("=" * 60)
print(f"  bronze_patients   : {patients_reloaded.count()} rows")
print(f"  bronze_admissions : {admissions_reloaded.count()} rows")
print(f"\n  Delta path: {DELTA_BASE}/bronze/")
print(f"\n  PII present in bronze_patients.patient_name")
print(f"  Will be masked in Silver layer")
print("=" * 60)
print("Bronze complete. No data was modified.")

BRONZE LAYER — HOSPITAL COMPLETE


  bronze_patients   : 55500 rows
  bronze_admissions : 55500 rows

  Delta path: /Users/malone/data-engineering-pathway/projects/hospital-lakehouse/delta/bronze/

  PII present in bronze_patients.patient_name
  Will be masked in Silver layer
Bronze complete. No data was modified.
